In [2]:
import celltypist
from celltypist import models
from scipy.stats import pearsonr
import seaborn as sns
import numpy as np
import scanpy as sc
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import concurrent.futures

In [8]:
expr_path = '/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/processed/WMB-10Xv3-Isocortex-1-raw-wmeta.h5ad'
adata = sc.read_h5ad(expr_path)
adata

AnnData object with n_obs × n_vars = 227670 × 32285
    obs: 'cell_barcode', 'barcoded_cell_sample_label', 'library_label', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'region_of_interest_acronym', 'donor_label', 'donor_genotype', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'region_of_interest_order', 'region_of_interest_color'
    var: 'gene_symbol'

In [8]:
# models.download_model_index()

📜 Retrieving model list from server https://celltypist.cog.sanger.ac.uk/models/models.json
📚 Total models in list: 54


In [9]:
# models.download_models(model="Mouse_Whole_Brain.pkl")

📂 Storing models in /home/momo/.celltypist/data/models
💾 Total models to download: 1
💾 Downloading model [1/1]: Mouse_Whole_Brain.pkl


In [11]:
# models.models_description().head()

👉 Detailed model information can be found at `https://www.celltypist.org/models`


,model,description
0,Immune_All_Low.pkl,immune sub-populations combined from 20 tissue...
1,Immune_All_High.pkl,immune populations combined from 20 tissues of...
2,Adult_COVID19_PBMC.pkl,peripheral blood mononuclear cell types from C...
3,Adult_CynomolgusMacaque_Hippocampus.pkl,cell types from the hippocampus of adult cynom...
4,Adult_Human_MTG.pkl,cell types and subtypes (10x-based) from the a...


In [10]:
model = models.Model.load(model="Mouse_Whole_Brain.pkl")
model

CellTypist model with 334 cell types and 5596 features
    date: 2024-04-30 17:19:37.721313
    details: cell types from the whole adult mouse brain
    source: https://doi.org/10.1038/s41586-023-06812-z
    version: v1
    cell types: 001 CLA-EPd-CTX Car3 Glut, 002 IT EP-CLA Glut, ..., 338 Lymphoid NN
    features: Xkr4, Rgs20, ..., mt-Cytb

In [11]:
# Process data before annotating
sc.pp.normalize_total(adata, target_sum=1e4)  # Normalize counts per cell
sc.pp.log1p(adata)  # Log-transform the data

In [12]:
adata.var.index = adata.var["gene_symbol"]  # Replace Ensembl IDs with gene symbols
adata.var.index = adata.var.index.astype(str)
# adata.var = adata.var[~adata.var.index.str.startswith("Blank")]  # Remove Blank genes

In [ ]:
predictions = celltypist.annotate(
    adata,
    model = "Mouse_Whole_Brain.pkl",
    use_GPU = False,
    majority_voting = True)

🔬 Input data has 227670 cells and 32285 genes
🔗 Matching reference genes in the model
🧬 5596 features used for prediction
⚖️ Scaling input data


In [9]:
predictions.predicted_labels.head()

,predicted_labels,over_clustering,majority_voting
cell_label,,,
GCACTAAGTACAAGTA-399_B02,018 L2 IT PPP-APr Glut,471,006 L4/5 IT CTX Glut
CAGAGCCGTCCTGGTG-399_A02,018 L2 IT PPP-APr Glut,365,006 L4/5 IT CTX Glut
CGCCATTCACGACAGA-403_A06,018 L2 IT PPP-APr Glut,365,006 L4/5 IT CTX Glut
GGGTTTATCCGATCGG-399_B02,018 L2 IT PPP-APr Glut,471,006 L4/5 IT CTX Glut
GTGTGATCAGACAAAT-399_B02,018 L2 IT PPP-APr Glut,432,020 L2/3 IT RSP Glut


In [10]:
# Load your dataframes (assuming they are named `predictions_df` and `metadata_df`)
merged_df = predictions.predicted_labels[["majority_voting"]].merge(adata.obs[['subclass']], left_index=True, right_index=True, how='inner')

# Display merged data
print(merged_df.head())

                               majority_voting                subclass
cell_label                                                            
GCACTAAGTACAAGTA-399_B02  006 L4/5 IT CTX Glut  018 L2 IT PPP-APr Glut
CAGAGCCGTCCTGGTG-399_A02  006 L4/5 IT CTX Glut  018 L2 IT PPP-APr Glut
CGCCATTCACGACAGA-403_A06  006 L4/5 IT CTX Glut  018 L2 IT PPP-APr Glut
GGGTTTATCCGATCGG-399_B02  006 L4/5 IT CTX Glut  018 L2 IT PPP-APr Glut
GTGTGATCAGACAAAT-399_B02  020 L2/3 IT RSP Glut  018 L2 IT PPP-APr Glut


In [11]:
accuracy = (merged_df['majority_voting'].astype(str) == merged_df['subclass'].astype(str)).mean()
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 96.18%


In [13]:
from sklearn.metrics import confusion_matrix

# Encode labels numerically
true_labels = merged_df['subclass']
predicted_labels = merged_df['majority_voting']

# Generate confusion matrix
conf_matrix = pd.crosstab(true_labels, predicted_labels, rownames=['True'], colnames=['Predicted'])
conf_matrix.head()

Predicted,001 CLA-EPd-CTX Car3 Glut,002 IT EP-CLA Glut,003 L5/6 IT TPE-ENT Glut,004 L6 IT CTX Glut,005 L5 IT CTX Glut,006 L4/5 IT CTX Glut,007 L2/3 IT CTX Glut,009 L2/3 IT PIR-ENTl Glut,010 IT AON-TT-DP Glut,020 L2/3 IT RSP Glut,...,327 Oligo NN,328 OEC NN,329 ABC NN,330 VLMC NN,331 Peri NN,332 SMC NN,333 Endo NN,334 Microglia NN,335 BAM NN,337 DC NN
True,,,,,,,,,,,,,,,,,,,,,
001 CLA-EPd-CTX Car3 Glut,1151,0,0,1,1,3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
002 IT EP-CLA Glut,1,973,32,105,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
003 L5/6 IT TPE-ENT Glut,0,80,516,141,53,235,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
004 L6 IT CTX Glut,0,182,69,8299,740,251,71,0,0,1,...,0,0,0,0,0,0,0,0,0,0
005 L5 IT CTX Glut,1,0,115,586,6782,1034,14,0,2,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(true_labels, predicted_labels)
print(f"Adjusted Rand Index: {ari:.3f}")

Adjusted Rand Index: 0.939


In [15]:
from sklearn.metrics import normalized_mutual_info_score

nmi = normalized_mutual_info_score(true_labels, predicted_labels)
print(f"Normalized Mutual Information: {nmi:.3f}")

Normalized Mutual Information: 0.936
